**PART 1.  WHAT IS AN EMBEDDING?**

- An embedding turns a piece of text into a list of numbers (a vector).  The model
is trained so that texts with similar MEANING get vectors that point in similar
directions.
 
   * "How many days of leave do I get?"   ->  [0.12, -0.40, 0.33, ... 384 numbers]
   * "Annual leave entitlement"           ->  [0.10, -0.38, 0.30, ...]   (close)
   * "Office parking rules"               ->  [-0.50, 0.21, 0.02, ...]   (far)
 
Compare with what you already know:
 
   - Day 19  Bag of Words / TF-IDF: counts WORDS.  "car" and "automobile" share no
                                    word, so their similarity is 0.

   - Day 19  nn.Embedding         : a table learned for ONE task (spam).

   - Today   sentence embeddings  : pre-trained on huge text, so "car" and
                                   "automobile" land close together, for any task.
 
Sizes you will meet:  all-MiniLM-L6-v2 gives 384 numbers per text; many API models
give 768 or 1536.  The size is called the DIMENSION.  It is fixed for a model.
 
*THE RULE THAT CAUSES MOST BUGS:*
   - Documents and queries MUST be embedded by the SAME model.  Different model,
   different space.  If the dimensions differ you get an error; if they happen to
   match, you get silent nonsense.

**PART 2.  MEASURING "CLOSE":  COSINE SIMILARITY**

- dot(a, b)  = a1*b1 + a2*b2 + ...
- |a|        = sqrt(a1^2 + a2^2 + ...)             the length of a
- cos(a, b)  = dot(a, b) / (|a| * |b|)             between -1 and 1

- +1  same direction (same meaning)      
-  0  unrelated (at right angles)
- -1  opposite direction

* Cosine ignores LENGTH, only direction matters: [1,1] and [2,2] have cosine 1.

*NORMALISING:  divide a vector by its own length, so its length becomes 1.
[3,4] has length 5, so normalised it is [0.6, 0.8].*

THE SHORTCUT EVERYONE USES:
- If both vectors are normalised, then   cos(a, b) = dot(a, b)
and the squared distance is            |a - b|^2 = 2 - 2 * cos(a, b)
So for normalised vectors, "highest dot product", "highest cosine" and "smallest
distance" all rank the documents in the SAME order.  That is why vector search
engines mostly just compute dot products.

**PART 3.  VECTOR SEARCH AND FAISS**

Semantic search, in 4 lines:
   1. Embed every document once (and normalise).         -> matrix E, one row each
   2. Embed the query the same way.                       -> vector q
   3. Scores = E @ q   (one dot product per document).
   4. Take the k highest scores  (top-k):  np.argsort(-scores)[:k]

- That is EXACT search (compare with everything).  It is fine up to about a million
vectors.  Beyond that, "approximate nearest neighbour" (ANN) indexes such as HNSW
or IVF skip most comparisons and trade a little accuracy for a lot of speed.


FAISS (Facebook AI Similarity Search) is a library that does steps 3 and 4 fast.
 
   import faiss

   index = faiss.IndexFlatIP(dim)            # exact search, inner product (= cosine if normalised)

   index.add(E.astype("float32"))            # E shape (n_docs, dim)

   scores, ids = index.search(q.reshape(1, -1).astype("float32"), k)*
   
   - scores and ids both have shape (1, k): one row per query
 
   - faiss.IndexFlatL2(dim) is the same but ranks by smaller distance.
   Things to remember:  FAISS wants float32, and the query must be 2-D (a row).
   FAISS stores ONLY vectors.  The ids are positions (0, 1, 2 ...), so YOU keep a
   list of the texts and metadata and map the position back.

**PART 4.  VECTOR DATABASES, METADATA AND FILTERING**

A vector database = vectors + the original text + metadata + saving to disk +
filtering, in one tool.
 
   - FAISS    a fast search LIBRARY.  Vectors only, you manage texts and files.
   - Chroma   a small database that runs in your own Python process and saves to a
            folder.  Easiest to start with.  (Hour 3 uses it.)
   - Qdrant   a database server, built for production: filters, many users.
 
Metadata is extra information stored with each chunk:

   * {"dept": "HR", "source": "handbook.pdf", "page": 12, "year": 2026}

You use it to FILTER:  "only HR documents", "only from 2026".

PRE-FILTER vs POST-FILTER   (a classic trap):
   - post-filter: take the top-k overall, THEN drop those that fail the filter.
                  You may be left with fewer than k results, even none.

   - pre-filter:  only search among documents that pass the filter, then take
                  the top-k.  You always get k results if k exist.
                  
   Real vector databases filter before ranking.  A hand-made FAISS script that
   filters afterwards has this bug.

**PART 5.  THE PIPELINE AND ITS LIMITS**

   * Documents -> split into chunks (Day 23) -> embed -> store in the vector DB
   * Question  -> embed (same model) -> search top-k -> return the chunks
 
Day 23 adds the last step: paste the retrieved chunks into the prompt and let the
LLM answer.  That whole thing is RAG, and it fixes hallucination because the model
now answers from text you gave it (Day 21, Q4).
 
Limits to remember:
   - "Similar" is not the same as "relevant".  "Leave policy" and "leave the
     building" can be close in a weak model.
   - Negation is weak:  "allowed" and "not allowed" often get similar vectors.
   - Exact things (IDs like INV-2026-105, names, numbers) are better found by
     plain keyword search.  Strong systems combine both (hybrid search).
   - Chunk size and top-k change the answer quality a lot (Day 23).

**Exercises**

E1 PAPER.  

a = [1, 2, 2],  b = [2, 0, 1].

   Compute dot(a, b), |a|, |b| and cos(a, b).  Round cos to 3 decimals.

   * (1x2) + (2x0) + (2x1) = 4
   * |a| = sqrt(1^2 + 2^2 + 2^2) = sqrt(1 + 4 + 4) = sqrt(9) = 3
   * |b| = sqrt(2^2 + 0^2 + 1^2) = sqrt(4 + 0 + 1) = sqrt(5) = 2.236
   * cos(a, b) = 4/(3x2.236) = 0.596

E2 PAPER.  Write the cosine (no calculation needed, think of the angle):

   (a)  [1, 0]  and  [0, 1]
   * 0

   (b)  [1, 1]  and  [2, 2]
   * 1

   (c)  [1, 0]  and  [-1, 0]
   * -1
 
E3  PAPER.  u = [1, 0] and v = [3, 4].

   (a)  Normalise v.
   * |v| = sqrt(3^2 + 4^2) = 5, normalise v: v/|v| = (3/5,4/5) = [0.6, 0.8]

   (b)  Compute cos(u, v) and then the squared distance |u - v_normalised|^2.
   * dot(u, v): (1x3) + (0) = 3, |u| = sqrt(1^1 + 0^2) = 1
   * cos(u, v) = 3/(5) = 0.6
   * u - v_normalised = [1 - 0.6, 0 - 0.8] = [0.4, -0.8]
   * |u - v_normalised|^2 = (0.4)^2 + (-0.8)^2 = 0.16 + 0.64 = 0.8

   (c)  Check that the squared distance equals 2 - 2 * cos(u, v_normalised).
   * cos(u, v_normalised) = cos(u, v) = 0.6
   * 2 - 2.cos(u, v_normalised) = 2 - 2(0.6) = 2 - 1.2 = 0.8 (checked)

In [7]:
import numpy as np
docs = ["leave policy", "annual leave days", "office parking",
            "dress code", "sick leave certificate"]
E = np.array([[0.9, 0.1, 0.0],
                  [0.8, 0.2, 0.1],
                  [0.0, 0.9, 0.3],
                  [0.1, 0.7, 0.6],
                  [0.7, 0.3, 0.2]])
q = np.array([0.85, 0.15, 0.05])
meta = [{"dept": "HR"}, {"dept": "HR"}, {"dept": "Facilities"}, {"dept": "Facilities"}, {"dept": "HR"}]

In [8]:
#E4  Exact search with NumPy
En = E / np.linalg.norm(E, axis=1, keepdims=True)
qn = q / np.linalg.norm(q)
scores = En @ qn
order = np.argsort(-scores)
print(scores.round(4))
print([docs[i] for i in order[:2]])

[0.9963 0.9956 0.1829 0.2744 0.9548]
['leave policy', 'annual leave days']


In [9]:
#E5  The same search with FAISS.
import faiss
index = faiss.IndexFlatIP(3)
index.add(En.astype("float32"))
S, I = index.search(qn.reshape(1, -1).astype("float32"), 2)
print((S,I, S.shape))


(array([[0.99628156, 0.9955794 ]], dtype=float32), array([[0, 1]]), (1, 2))


**Question**
- Do E4 and E5 match? Why does inner product give the cosine here?
   * Yes, because the inner product does the same cosine because of the normalization. FAISS’s IndexFlatIP is only a cosine search because you normalised the vectors first. If you forget to normalise, you get raw dot products, which are biased towards longer vectors.

**E6  Metadata.  The user only wants Facilities documents.**

(a) POST-FILTER:  take the top-2 from E4, then keep only the Facilities ones. What do you get?
   * Nothing because the top-2 overall documents are Index 0 ("leave policy") and Index 1 ("annual leave days"), both of which belong to the HR department. When we filter them afterward to keep only Facilities documents, nothing remains.

(b) PRE-FILTER:  keep only the Facilities rows first, rank those, take the top-2. What do you get, in order, with scores?
   * By filtering out non-Facilities documents first, we isolate only Index 2 ("office parking") and Index 3 ("dress code"). Ranking just those two gives "dress code" (0.274) first and "office parking" (0.183) second.

(c) One sentence: why is (a) a bug?
   *  Post-filtering is a bug because if all top-$k$ global results belong to excluded metadata categories, the system returns zero valid results, discarding relevant documents that met the metadata criteria.

**E7  Same dimension, different model.  Your documents were embedded with model A and a colleague embeds the query with model B, which also has 3 dimensions.**

(a) Will FAISS raise an error?  Is the answer trustworthy?  And what happens if model B had 4 dimensions?
   * No, because Faiss search always looks for the vector number but it is not trustworthy because both are in different model, and performing dot product on different model is meaningless.
   * It will raise error because it will mismatch the shape.

E8  In two or three lines each:
(a)  Why can a sentence embedding find "automobile" for a query "car", while TF-IDF with cosine cannot?
   * because the TF-IDF counts words "car" and "automobile" so cosine is 0. An embedding model will see both words
   and the model was trained on huge text, where “car” and “automobile” appear in similar contexts, so it places them close together.

(b)  Why is a normalised dot product equal to cosine?
   * because mathematical formula for cosine similarity is the dot product divided by the product of the two vectors' lengths.
   Normalisation scales vectors down to a length of 1. When |a| = 1 and |b| = 1, 
   the denominator becomes 1.1 = 1, leaving the dot product equal to the cosine.

E9  What do these two lines do wrong?  Name the problem in each.
 
        index.search(qn.astype("float32"), 2)                 # qn has shape (3,)
        index.search(np.zeros((1, 4), dtype="float32"), 2)    # the index has dimension 3

   * the first line missing reshape(1, -1). qn is 1-D, shape (3,), and FAISS needs 2-D (n_queries, dim).
   * the second line mismatch shape since the index has dimension 3.

E10 Choose and explain in one line each:  FAISS, Chroma or Qdrant?
(a)  A notebook that searches 5,000 chunks, rebuilt every time.
   * Faiss: a plain in-memory index that is quick to rebuild, with no need to keep anything.

(b)  A small app that must keep its chunks after a restart, one user.
   * Chroma: it runs inside your own process, saves to a folder, and keeps the text and metadata with the vectors.

(c)  A company service with many users and strict filters on department and year.
   * Qdrant: it is a server built for many users, with strong metadata filtering and production features.

**4-line semantic search from memory, with the notebook closed.**

1. normalize the document
   * En = E / np.linalg.norm(E, axis=1, keepdims=True)

2. normalize the query vector
   * qn = q / np.linalg.norm(q)

3. compute cosine similarity
   * scores = En @ qn

4. get the top-k
   * top_indices = np.argsort(scores)[::-1][:k]
